# 03 - Modelagem da Camada Gold

Este notebook realiza a modelagem analítica da camada Gold do pipeline.

A partir das tabelas Silver, os dados serão organizados em uma estrutura dimensional voltada às perguntas de negócio do MVP, com separação entre tabela fato e dimensões.

A modelagem busca facilitar consultas analíticas por companhia aérea, aeroporto, rota e tempo, preservando a rastreabilidade dos dados tratados nas etapas anteriores.

In [0]:
df_vra = spark.table("workspace.mvp_sprint_3_anac.silver_vra")
df_aerodromos = spark.table("workspace.mvp_sprint_3_anac.silver_aerodromos")

print("VRA Silver:", df_vra.count())
print("Aeródromos Silver:", df_aerodromos.count())

VRA Silver: 591431
Aeródromos Silver: 6873


In [0]:
dim_companhia = (
    df_vra
    .select(
        "sigla_icao_empresa_aerea",
        "empresa_aerea"
    )
    .dropDuplicates()
)

display(dim_companhia)

sigla_icao_empresa_aerea,empresa_aerea
LCO,LAN CARGO S.A.
PLS,PLACAR LINHAS AÉREAS S.A.
QTR,QATAR AIRWAYS GROUP
SAA,SOUTH AFRICAN AIRWAYS STATE OWNED COMPANY (SOC) LIMITED
SKX,SKY AIRLINE PERU S.A.C.
SLM,SURINAM AIRWAYS LTD
THY,TURKISH AIRLINES INC
AEB,AVION EXPRESS BRASIL LTDA
ARG,AEROLINEAS ARGENTINAS S/A
CHZ,CHALLENGE AIRCARGO LTD.


In [0]:
from pyspark.sql.functions import (
    col,
    date_format,
    dayofmonth,
    dayofweek,
    month,
    year
)

dim_tempo = (
    df_vra
    .select(
        col("referencia_ts").cast("date").alias("data")
    )
    .filter(col("data").isNotNull())
    .dropDuplicates()
    .withColumn("ano", year(col("data")))
    .withColumn("mes", month(col("data")))
    .withColumn("dia", dayofmonth(col("data")))
    .withColumn("dia_semana_num", dayofweek(col("data")))
    .withColumn("dia_semana", date_format(col("data"), "EEEE"))
)

display(dim_tempo)

data,ano,mes,dia,dia_semana_num,dia_semana
2026-02-12,2026,2,12,5,Thursday
2026-04-22,2026,4,22,4,Wednesday
2026-03-29,2026,3,29,1,Sunday
2026-07-11,2026,7,11,7,Saturday
2026-05-02,2026,5,2,7,Saturday
2026-03-04,2026,3,4,4,Wednesday
2026-07-25,2026,7,25,7,Saturday
2026-06-26,2026,6,26,6,Friday
2026-07-18,2026,7,18,7,Saturday
2026-05-08,2026,5,8,6,Friday


In [0]:
from pyspark.sql.functions import lit

aeroportos_vra = (
    df_vra
    .select(
        col("sigla_icao_aeroporto_origem").alias("codigo_oaci"),
        col("descricao_aeroporto_origem").alias("descricao_aeroporto")
    )
    .union(
        df_vra.select(
            col("sigla_icao_aeroporto_destino").alias("codigo_oaci"),
            col("descricao_aeroporto_destino").alias("descricao_aeroporto")
        )
    )
    .filter(col("codigo_oaci").isNotNull())
    .dropDuplicates()
)

dim_aeroporto = (
    aeroportos_vra
    .join(
        df_aerodromos.select(
            "codigo_oaci",
            "nome",
            "municipio",
            "uf",
            "pais",
            "municipio_servido",
            "uf_servido",
            "latitude_double",
            "longitude_double",
            "altitude_double"
        ),
        on="codigo_oaci",
        how="left"
    )
)

display(dim_aeroporto)

codigo_oaci,descricao_aeroporto,nome,municipio,uf,pais,municipio_servido,uf_servido,latitude_double,longitude_double,altitude_double
SBSV,DEPUTADO LUÍS EDUARDO MAGALHÃES - SALVADOR - BA - BRASIL,DEPUTADO LUÍS EDUARDO MAGALHÃES,SALVADOR,BA,Brasil,Salvador,BA,12.903333333333334,38.327222222222225,20.0
SPZO,"SUBTENIENTE FAP ALEJANDRO VELASCO ASTETE INTERNATIONAL AIRPORT - CUSCO, CUSCO REGION - PERU",null,null,null,null,null,null,null,null,null
SNHS,SANTA MAGALHÃES - SERRA TALHADA - PE - BRASIL,Santa Magalhães,SERRA TALHADA,PE,Brasil,Serra Talhada,PE,8.052222222222223,38.327222222222225,470.0
SBJU,ORLANDO BEZERRA DE MENEZES - JUAZEIRO DO NORTE - CE - BRASIL,Orlando Bezerra de Menezes,JUAZEIRO DO NORTE,CE,Brasil,Juazeiro do Norte,CE,7.2186111111111115,39.277499999999996,409.0
SBGV,CORONEL ALTINO MACHADO - GOVERNADOR VALADARES - MG - BRASIL,Coronel Altino Machado,GOVERNADOR VALADARES,MG,Brasil,Governador Valadares,MG,18.888333333333332,41.99472222222222,171.0
MPPA,"PANAMA PACIFICO, FORMERLY HOWARD AIR FORCE BASE - BALBOA - PANAMÁ",null,null,null,null,null,null,null,null,null
SBCT,AFONSO PENA - SÃO JOSÉ DOS PINHAIS - PR - BRASIL,Afonso Pena,CURITIBA,PR,Brasil,Curitiba,PR,25.52361111111111,49.180277777777775,911.0
SBCH,SERAFIN ENOSS BERTASO - CHAPECÓ - SC - BRASIL,Serafin Enoss Bertaso,CHAPECÓ,SC,Brasil,Chapecó,SC,27.140833333333333,52.66444444444444,654.0
SBGO,SANTA GENOVEVA/GOIÂNIA - GOIÂNIA - GO - BRASIL,SANTA GENOVEVA/GOIÂNIA,GOIÂNIA,GO,Brasil,Goiânia,GO,16.621111111111112,49.23027777777778,747.0
SBPP,PONTA PORÃ - PONTA PORÃ - MS - BRASIL,Aeroporto Internacional de Ponta Porã,PONTA PORÃ,MS,Brasil,Ponta Porã,MS,22.539444444444445,55.71527777777778,657.0


In [0]:
from pyspark.sql.functions import concat_ws

dim_rota = (
    df_vra
    .select(
        col("sigla_icao_aeroporto_origem").alias("aeroporto_origem"),
        col("sigla_icao_aeroporto_destino").alias("aeroporto_destino")
    )
    .filter(
        col("aeroporto_origem").isNotNull() &
        col("aeroporto_destino").isNotNull()
    )
    .dropDuplicates()
    .withColumn(
        "rota_id",
        concat_ws(
            "->",
            col("aeroporto_origem"),
            col("aeroporto_destino")
        )
    )
)

display(dim_rota)

aeroporto_origem,aeroporto_destino,rota_id
GVAC,SBKP,GVAC->SBKP
EDDF,GVAC,EDDF->GVAC
SBKP,SCEL,SBKP->SCEL
EBBR,EDDF,EBBR->EDDF
SCEL,SBGR,SCEL->SBGR
SBGR,SKBO,SBGR->SKBO
SKBO,KMIA,SKBO->KMIA
SBGL,SCEL,SBGL->SCEL
SBKP,SBGL,SBKP->SBGL
LEMD,GVAC,LEMD->GVAC


In [0]:
print("Companhias:", dim_companhia.count())
print("Datas:", dim_tempo.count())
print("Aeroportos:", dim_aeroporto.count())
print("Rotas:", dim_rota.count())

Companhias: 115
Datas: 212
Aeroportos: 360
Rotas: 2649


In [0]:
from pyspark.sql.functions import count

duplicados_companhia = (
    dim_companhia
    .groupBy("sigla_icao_empresa_aerea")
    .agg(count("*").alias("quantidade"))
    .filter(col("quantidade") > 1)
    .orderBy(col("quantidade").desc())
)

display(duplicados_companhia)

sigla_icao_empresa_aerea,quantidade


In [0]:
duplicados_aeroporto = (
    dim_aeroporto
    .groupBy("codigo_oaci")
    .agg(count("*").alias("quantidade"))
    .filter(col("quantidade") > 1)
    .orderBy(col("quantidade").desc())
)

display(duplicados_aeroporto)

codigo_oaci,quantidade


In [0]:
print(
    "Datas duplicadas:",
    dim_tempo.count() - dim_tempo.select("data").distinct().count()
)

print(
    "Rotas duplicadas:",
    dim_rota.count() -
    dim_rota.select(
        "aeroporto_origem",
        "aeroporto_destino"
    ).distinct().count()
)

Datas duplicadas: 0
Rotas duplicadas: 0


In [0]:
from pyspark.sql.functions import concat_ws

fato_voo = (
    df_vra
    .select(
        "sigla_icao_empresa_aerea",
        "numero_voo",
        "codigo_di",
        "codigo_tipo_linha",
        "modelo_equipamento",
        "numero_de_assentos_int",

        col("sigla_icao_aeroporto_origem").alias("codigo_aeroporto_origem"),
        col("sigla_icao_aeroporto_destino").alias("codigo_aeroporto_destino"),

        col("referencia_ts").cast("date").alias("data"),

        "partida_prevista_ts",
        "partida_real_ts",
        "chegada_prevista_ts",
        "chegada_real_ts",

        "atraso_partida_min",
        "atraso_chegada_min",

        "flag_atraso_30",
        "flag_atraso_60",
        "flag_cancelado",
        "flag_atraso_extremo",

        "situacao_voo",
        "situacao_partida",
        "situacao_chegada",

        "hora_partida_prevista",
        "periodo_dia"
    )
    .withColumn(
        "rota_id",
        concat_ws(
            "->",
            col("codigo_aeroporto_origem"),
            col("codigo_aeroporto_destino")
        )
    )
)

In [0]:
print("Silver VRA:", df_vra.count())
print("Fato voo:", fato_voo.count())

display(fato_voo)

Silver VRA: 591431
Fato voo: 591431


sigla_icao_empresa_aerea,numero_voo,codigo_di,codigo_tipo_linha,modelo_equipamento,numero_de_assentos_int,codigo_aeroporto_origem,codigo_aeroporto_destino,data,partida_prevista_ts,partida_real_ts,chegada_prevista_ts,chegada_real_ts,atraso_partida_min,atraso_chegada_min,flag_atraso_30,flag_atraso_60,flag_cancelado,flag_atraso_extremo,situacao_voo,situacao_partida,situacao_chegada,hora_partida_prevista,periodo_dia,rota_id
LCO,Z1517,1,G,B763,0,GVAC,SBKP,2026-03-01,null,2026-03-02T09:54:00.000Z,null,2026-03-02T16:36:00.000Z,null,null,null,null,0,null,REALIZADO,null,null,null,Não informado,GVAC->SBKP
LCO,Z1517,1,G,B763,0,EDDF,GVAC,2026-03-01,null,2026-03-01T16:19:00.000Z,null,2026-03-01T22:32:00.000Z,null,null,null,null,0,null,REALIZADO,null,null,null,Não informado,EDDF->GVAC
LCO,Z1517,1,G,B763,0,SBKP,SCEL,2026-03-01,null,2026-03-02T17:37:00.000Z,null,2026-03-02T21:28:00.000Z,null,null,null,null,0,null,REALIZADO,null,null,null,Não informado,SBKP->SCEL
LCO,Z1517,1,G,B763,0,EBBR,EDDF,2026-03-01,null,2026-03-01T13:59:00.000Z,null,2026-03-01T15:15:00.000Z,null,null,null,null,0,null,REALIZADO,null,null,null,Não informado,EBBR->EDDF
LCO,Z1600,1,G,B763,0,SCEL,SBGR,2026-03-01,null,2026-03-01T04:31:00.000Z,null,2026-03-01T08:59:00.000Z,null,null,null,null,0,null,REALIZADO,null,null,null,Não informado,SCEL->SBGR
LCO,Z1600,1,G,B763,0,SBGR,SKBO,2026-03-01,null,2026-03-01T09:57:00.000Z,null,2026-03-01T15:46:00.000Z,null,null,null,null,0,null,REALIZADO,null,null,null,Não informado,SBGR->SKBO
LCO,Z1600,1,G,B763,0,SKBO,KMIA,2026-03-01,null,2026-03-01T17:12:00.000Z,null,2026-03-01T20:57:00.000Z,null,null,null,null,0,null,REALIZADO,null,null,null,Não informado,SKBO->KMIA
LCO,Z3616,1,G,B763,0,SBGL,SCEL,2026-03-01,null,2026-03-03T08:44:00.000Z,null,2026-03-03T13:34:00.000Z,null,null,null,null,0,null,REALIZADO,null,null,null,Não informado,SBGL->SCEL
LCO,Z3616,1,G,B763,0,SBKP,SBGL,2026-03-01,null,2026-03-03T06:29:00.000Z,null,2026-03-03T07:28:00.000Z,null,null,null,null,0,null,REALIZADO,null,null,null,Não informado,SBKP->SBGL
LCO,Z3616,1,G,B763,0,LEMD,GVAC,2026-03-01,null,2026-03-01T11:22:00.000Z,null,2026-03-01T16:11:00.000Z,null,null,null,null,0,null,REALIZADO,null,null,null,Não informado,LEMD->GVAC


In [0]:
companhias_sem_dim = (
    fato_voo
    .select("sigla_icao_empresa_aerea")
    .distinct()
    .join(
        dim_companhia.select("sigla_icao_empresa_aerea"),
        on="sigla_icao_empresa_aerea",
        how="left_anti"
    )
    .count()
)

datas_sem_dim = (
    fato_voo
    .select("data")
    .filter(col("data").isNotNull())
    .distinct()
    .join(
        dim_tempo.select("data"),
        on="data",
        how="left_anti"
    )
    .count()
)

rotas_sem_dim = (
    fato_voo
    .select("rota_id")
    .distinct()
    .join(
        dim_rota.select("rota_id"),
        on="rota_id",
        how="left_anti"
    )
    .count()
)

print("Companhias da fato sem dimensão:", companhias_sem_dim)
print("Datas da fato sem dimensão:", datas_sem_dim)
print("Rotas da fato sem dimensão:", rotas_sem_dim)

Companhias da fato sem dimensão: 0
Datas da fato sem dimensão: 0
Rotas da fato sem dimensão: 0


In [0]:
dim_companhia.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_sprint_3_anac.gold_dim_companhia")

In [0]:
dim_tempo.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_sprint_3_anac.gold_dim_tempo")

In [0]:
dim_aeroporto.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_sprint_3_anac.gold_dim_aeroporto")

In [0]:
dim_rota.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_sprint_3_anac.gold_dim_rota")

In [0]:
fato_voo.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_sprint_3_anac.gold_fato_voo")

In [0]:
print("gold_dim_companhia:", spark.table("workspace.mvp_sprint_3_anac.gold_dim_companhia").count())
print("gold_dim_tempo:", spark.table("workspace.mvp_sprint_3_anac.gold_dim_tempo").count())
print("gold_dim_aeroporto:", spark.table("workspace.mvp_sprint_3_anac.gold_dim_aeroporto").count())
print("gold_dim_rota:", spark.table("workspace.mvp_sprint_3_anac.gold_dim_rota").count())
print("gold_fato_voo:", spark.table("workspace.mvp_sprint_3_anac.gold_fato_voo").count())

gold_dim_companhia: 115
gold_dim_tempo: 212
gold_dim_aeroporto: 360
gold_dim_rota: 2649
gold_fato_voo: 591431


In [0]:
display(
    spark.sql(
        "SHOW TABLES IN workspace.mvp_sprint_3_anac"
    )
)

database,tableName,isTemporary
mvp_sprint_3_anac,bronze_aerodromos,false
mvp_sprint_3_anac,bronze_vra,false
mvp_sprint_3_anac,gold_dim_aeroporto,false
mvp_sprint_3_anac,gold_dim_companhia,false
mvp_sprint_3_anac,gold_dim_rota,false
mvp_sprint_3_anac,gold_dim_tempo,false
mvp_sprint_3_anac,gold_fato_voo,false
mvp_sprint_3_anac,silver_aerodromos,false
mvp_sprint_3_anac,silver_vra,false


## Resultado da modelagem Gold

A camada Gold foi estruturada em modelo dimensional para facilitar as análises definidas no objetivo do MVP.

Foram criadas as seguintes tabelas:

- `gold_fato_voo`: tabela fato cuja granularidade corresponde a um registro operacional do VRA.
- `gold_dim_companhia`: dimensão com códigos ICAO e nomes das companhias aéreas.
- `gold_dim_tempo`: dimensão de datas utilizada nas análises temporais.
- `gold_dim_aeroporto`: dimensão consolidada de aeroportos, preservando códigos presentes no VRA e enriquecendo-os com dados cadastrais da ANAC quando disponíveis.
- `gold_dim_rota`: dimensão com as combinações de aeroportos de origem e destino.

A dimensão de aeroportos é utilizada como uma dimensão do tipo *role-playing*, atuando tanto como aeroporto de origem quanto como aeroporto de destino.

As validações realizadas confirmaram que a tabela fato manteve os 591.431 registros da camada Silver e que todas as companhias, datas e rotas presentes na fato possuem correspondência em suas respectivas dimensões.